# IF 2D por registradores e por coluna

Este notebook implementa a versão **2D nested** da Inspection Factorization (IF), usando as matrizes 1D:

\[
A^T =
\begin{bmatrix}
1&0&0&1&1&0\\
0&1&0&1&0&1\\
0&0&1&0&1&1
\end{bmatrix}
\]

\[
B =
\begin{bmatrix}
1&0&0\\
0&1&0\\
0&0&1\\
1&1&0\\
1&0&1\\
0&1&1
\end{bmatrix}
\]

\[
C^T =
\begin{bmatrix}
1&-1&-1&0&0\\
0&-1&1&-1&0\\
0&0&-1&-1&1\\
0&1&0&0&0\\
0&0&1&0&0\\
0&0&0&1&0
\end{bmatrix}
\]

O fluxo 2D é:

\[
D = C^T d C
\]

\[
G = B g B^T
\]

\[
S = D \odot G
\]

\[
s = A^T S A
\]

com:

\[
d \in \mathbb{R}^{5\times5}, \quad g \in \mathbb{R}^{3\times3},
\quad D,G,S \in \mathbb{R}^{6\times6}, \quad s \in \mathbb{R}^{3\times3}.
\]

A implementação principal gera uma coluna de \(D\) por vez e acumula diretamente a saída, sem materializar \(D\) nem \(S\) inteiros.


In [1]:
from fractions import Fraction as F
import numpy as np


## 1. Matrizes de referência

Estas matrizes são usadas para validação com NumPy.


In [2]:
AT = np.array([
    [1, 0, 0, 1, 1, 0],
    [0, 1, 0, 1, 0, 1],
    [0, 0, 1, 0, 1, 1],
], dtype=object)

B = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
], dtype=object)

CT = np.array([
    [1, -1, -1,  0, 0],
    [0, -1,  1, -1, 0],
    [0,  0, -1, -1, 1],
    [0,  1,  0,  0, 0],
    [0,  0,  1,  0, 0],
    [0,  0,  0,  1, 0],
], dtype=object)

A = AT.T
C = CT.T

print("AT shape:", AT.shape)
print("A shape: ", A.shape)
print("B shape: ", B.shape)
print("CT shape:", CT.shape)
print("C shape: ", C.shape)


AT shape: (3, 6)
A shape:  (6, 3)
B shape:  (6, 3)
CT shape: (6, 5)
C shape:  (5, 6)


## 2. Entradas editáveis e referência direta

Os valores abaixo são apenas um exemplo para teste.  
Você pode editar `d_np` e `g_np`.


In [ ]:
d_np = np.array([
    [ 0,  1,  2,  3,  4],
    [ 5,  6,  7,  8,  9],
    [10, 11, 12, 13, 14],
    [15, 16, 17, 18, 19],
    [20, 21, 22, 23, 24],
], dtype=object)

g_np = np.array([
    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8],
], dtype=object)

D_ref = CT @ d_np @ C
G_ref = B @ g_np @ B.T
S_ref = D_ref * G_ref
s_ref = AT @ S_ref @ A

print("D_ref =")
print(D_ref)
print()
print("G_ref =")
print(G_ref)
print()
print("S_ref =")
print(S_ref)
print()
print("s_ref =")
print(s_ref)


## 3. Implementação por registradores

A célula abaixo é específica para as matrizes IF 2D deste notebook.

### Registradores do datapath online, assumindo \(G\) pré-computado

- `d00..d44`: 25 registradores para o tile \(d[5\times5]\).
- `acc0..acc5`: 6 registradores para uma coluna de \(D\).
- `y0..y2`: 3 registradores para \(y=A^T S_{:,j}\).
- `s00..s22`: 9 registradores para a saída \(s[3\times3]\).

Total sem contar \(G\):

\[
25 + 6 + 3 + 9 = 43
\]

Se \(G\) também for armazenado em registradores:

\[
43 + 36 = 79
\]


# ============================================================
# REGISTRADORES DO FILTRO ORIGINAL g[3x3]
# Propósito: armazenar o kernel antes da transformação.
# ============================================================
g00 = F(0)
g01 = F(1)
g02 = F(2)

g10 = F(3)
g11 = F(4)
g12 = F(5)

g20 = F(6)
g21 = F(7)
g22 = F(8)

# ============================================================
# REGISTRADORES DO FILTRO TRANSFORMADO G[6x6] = B @ g @ B.T
# Em hardware, G pode ser pré-computado offline e lido de ROM/SRAM.
# ============================================================
G00 = g00
G01 = g01
G02 = g02
G03 = g00 + g01
G04 = g00 + g02
G05 = g01 + g02

G10 = g10
G11 = g11
G12 = g12
G13 = g10 + g11
G14 = g10 + g12
G15 = g11 + g12

G20 = g20
G21 = g21
G22 = g22
G23 = g20 + g21
G24 = g20 + g22
G25 = g21 + g22

G30 = g00 + g10
G31 = g01 + g11
G32 = g02 + g12
G33 = g00 + g01 + g10 + g11
G34 = g00 + g02 + g10 + g12
G35 = g01 + g02 + g11 + g12

G40 = g00 + g20
G41 = g01 + g21
G42 = g02 + g22
G43 = g00 + g01 + g20 + g21
G44 = g00 + g02 + g20 + g22
G45 = g01 + g02 + g21 + g22

G50 = g10 + g20
G51 = g11 + g21
G52 = g12 + g22
G53 = g10 + g11 + g20 + g21
G54 = g10 + g12 + g20 + g22
G55 = g11 + g12 + g21 + g22


In [3]:
# ============================================================
# REGISTRADORES DE ENTRADA d[5x5]
# Propósito: armazenar o tile de entrada usado pelo IF 2D.
# ============================================================
d00 = F(0)
d01 = F(1)
d02 = F(2)
d03 = F(3)
d04 = F(4)

d10 = F(5)
d11 = F(6)
d12 = F(7)
d13 = F(8)
d14 = F(9)

d20 = F(10)
d21 = F(11)
d22 = F(12)
d23 = F(13)
d24 = F(14)

d30 = F(15)
d31 = F(16)
d32 = F(17)
d33 = F(18)
d34 = F(19)

d40 = F(20)
d41 = F(21)
d42 = F(22)
d43 = F(23)
d44 = F(24)



# ============================================================
# REGISTRADORES DA COLUNA ATUAL DE D
# acc0..acc5 representam D[0,j]..D[5,j].
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# ============================================================
# REGISTRADORES INTERMEDIÁRIOS y = A.T @ S[:,j]
# y0..y2 são reutilizados para cada coluna j.
# ============================================================
y0 = F(0)
y1 = F(0)
y2 = F(0)

# ============================================================
# REGISTRADORES DE SAÍDA s[3x3]
# Propósito: acumular o resultado final s = A.T @ S @ A.
# ============================================================
s00 = F(0)
s01 = F(0)
s02 = F(0)

s10 = F(0)
s11 = F(0)
s12 = F(0)

s20 = F(0)
s21 = F(0)
s22 = F(0)




# ============================================================
# COLUNA 0 DE D
# C[:,0] = [1, -1, -1, 0, 0]
# A[0,:] = [1, 0, 0]
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# t0 = d[0,:] @ C[:,0] = d00 - d01 - d02
acc0 += (d00 - d01 - d02)

# t1 = d[1,:] @ C[:,0] = d10 - d11 - d12
acc0 -= (d10 - d11 - d12)
acc1 -= (d10 - d11 - d12)
acc3 += (d10 - d11 - d12)

# t2 = d[2,:] @ C[:,0] = d20 - d21 - d22
acc0 -= (d20 - d21 - d22)
acc1 += (d20 - d21 - d22)
acc2 -= (d20 - d21 - d22)
acc4 += (d20 - d21 - d22)

# t3 = d[3,:] @ C[:,0] = d30 - d31 - d32
acc1 -= (d30 - d31 - d32)
acc2 -= (d30 - d31 - d32)
acc5 += (d30 - d31 - d32)

# t4 = d[4,:] @ C[:,0] = d40 - d41 - d42
acc2 += (d40 - d41 - d42)

# Agora acc0..acc5 = D[:,0]
# Calcula y = A.T @ (D[:,0] ⊙ G[:,0])
y0 = F(0)
y1 = F(0)
y2 = F(0)
y0 += acc0 * G00
y0 += acc3 * G30
y0 += acc4 * G40
y1 += acc1 * G10
y1 += acc3 * G30
y1 += acc5 * G50
y2 += acc2 * G20
y2 += acc4 * G40
y2 += acc5 * G50

# Acumula s += y @ A[0,:]
s00 += y0
s10 += y1
s20 += y2


# ============================================================
# COLUNA 1 DE D
# C[:,1] = [0, -1, 1, -1, 0]
# A[1,:] = [0, 1, 0]
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# t0 = d[0,:] @ C[:,1] = -d01 + d02 - d03
acc0 += (-d01 + d02 - d03)

# t1 = d[1,:] @ C[:,1] = -d11 + d12 - d13
acc0 -= (-d11 + d12 - d13)
acc1 -= (-d11 + d12 - d13)
acc3 += (-d11 + d12 - d13)

# t2 = d[2,:] @ C[:,1] = -d21 + d22 - d23
acc0 -= (-d21 + d22 - d23)
acc1 += (-d21 + d22 - d23)
acc2 -= (-d21 + d22 - d23)
acc4 += (-d21 + d22 - d23)

# t3 = d[3,:] @ C[:,1] = -d31 + d32 - d33
acc1 -= (-d31 + d32 - d33)
acc2 -= (-d31 + d32 - d33)
acc5 += (-d31 + d32 - d33)

# t4 = d[4,:] @ C[:,1] = -d41 + d42 - d43
acc2 += (-d41 + d42 - d43)

# Agora acc0..acc5 = D[:,1]
# Calcula y = A.T @ (D[:,1] ⊙ G[:,1])
y0 = F(0)
y1 = F(0)
y2 = F(0)
y0 += acc0 * G01
y0 += acc3 * G31
y0 += acc4 * G41
y1 += acc1 * G11
y1 += acc3 * G31
y1 += acc5 * G51
y2 += acc2 * G21
y2 += acc4 * G41
y2 += acc5 * G51

# Acumula s += y @ A[1,:]
s01 += y0
s11 += y1
s21 += y2


# ============================================================
# COLUNA 2 DE D
# C[:,2] = [0, 0, -1, -1, 1]
# A[2,:] = [0, 0, 1]
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# t0 = d[0,:] @ C[:,2] = -d02 - d03 + d04
acc0 += (-d02 - d03 + d04)

# t1 = d[1,:] @ C[:,2] = -d12 - d13 + d14
acc0 -= (-d12 - d13 + d14)
acc1 -= (-d12 - d13 + d14)
acc3 += (-d12 - d13 + d14)

# t2 = d[2,:] @ C[:,2] = -d22 - d23 + d24
acc0 -= (-d22 - d23 + d24)
acc1 += (-d22 - d23 + d24)
acc2 -= (-d22 - d23 + d24)
acc4 += (-d22 - d23 + d24)

# t3 = d[3,:] @ C[:,2] = -d32 - d33 + d34
acc1 -= (-d32 - d33 + d34)
acc2 -= (-d32 - d33 + d34)
acc5 += (-d32 - d33 + d34)

# t4 = d[4,:] @ C[:,2] = -d42 - d43 + d44
acc2 += (-d42 - d43 + d44)

# Agora acc0..acc5 = D[:,2]
# Calcula y = A.T @ (D[:,2] ⊙ G[:,2])
y0 = F(0)
y1 = F(0)
y2 = F(0)
y0 += acc0 * G02
y0 += acc3 * G32
y0 += acc4 * G42
y1 += acc1 * G12
y1 += acc3 * G32
y1 += acc5 * G52
y2 += acc2 * G22
y2 += acc4 * G42
y2 += acc5 * G52

# Acumula s += y @ A[2,:]
s02 += y0
s12 += y1
s22 += y2


# ============================================================
# COLUNA 3 DE D
# C[:,3] = [0, 1, 0, 0, 0]
# A[3,:] = [1, 1, 0]
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# t0 = d[0,:] @ C[:,3] = d01
acc0 += (d01)

# t1 = d[1,:] @ C[:,3] = d11
acc0 -= (d11)
acc1 -= (d11)
acc3 += (d11)

# t2 = d[2,:] @ C[:,3] = d21
acc0 -= (d21)
acc1 += (d21)
acc2 -= (d21)
acc4 += (d21)

# t3 = d[3,:] @ C[:,3] = d31
acc1 -= (d31)
acc2 -= (d31)
acc5 += (d31)

# t4 = d[4,:] @ C[:,3] = d41
acc2 += (d41)

# Agora acc0..acc5 = D[:,3]
# Calcula y = A.T @ (D[:,3] ⊙ G[:,3])
y0 = F(0)
y1 = F(0)
y2 = F(0)
y0 += acc0 * G03
y0 += acc3 * G33
y0 += acc4 * G43
y1 += acc1 * G13
y1 += acc3 * G33
y1 += acc5 * G53
y2 += acc2 * G23
y2 += acc4 * G43
y2 += acc5 * G53

# Acumula s += y @ A[3,:]
s00 += y0
s01 += y0
s10 += y1
s11 += y1
s20 += y2
s21 += y2


# ============================================================
# COLUNA 4 DE D
# C[:,4] = [0, 0, 1, 0, 0]
# A[4,:] = [1, 0, 1]
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# t0 = d[0,:] @ C[:,4] = d02
acc0 += (d02)

# t1 = d[1,:] @ C[:,4] = d12
acc0 -= (d12)
acc1 -= (d12)
acc3 += (d12)

# t2 = d[2,:] @ C[:,4] = d22
acc0 -= (d22)
acc1 += (d22)
acc2 -= (d22)
acc4 += (d22)

# t3 = d[3,:] @ C[:,4] = d32
acc1 -= (d32)
acc2 -= (d32)
acc5 += (d32)

# t4 = d[4,:] @ C[:,4] = d42
acc2 += (d42)

# Agora acc0..acc5 = D[:,4]
# Calcula y = A.T @ (D[:,4] ⊙ G[:,4])
y0 = F(0)
y1 = F(0)
y2 = F(0)
y0 += acc0 * G04
y0 += acc3 * G34
y0 += acc4 * G44
y1 += acc1 * G14
y1 += acc3 * G34
y1 += acc5 * G54
y2 += acc2 * G24
y2 += acc4 * G44
y2 += acc5 * G54

# Acumula s += y @ A[4,:]
s00 += y0
s02 += y0
s10 += y1
s12 += y1
s20 += y2
s22 += y2


# ============================================================
# COLUNA 5 DE D
# C[:,5] = [0, 0, 0, 1, 0]
# A[5,:] = [0, 1, 1]
# ============================================================
acc0 = F(0)
acc1 = F(0)
acc2 = F(0)
acc3 = F(0)
acc4 = F(0)
acc5 = F(0)

# t0 = d[0,:] @ C[:,5] = d03
acc0 += (d03)

# t1 = d[1,:] @ C[:,5] = d13
acc0 -= (d13)
acc1 -= (d13)
acc3 += (d13)

# t2 = d[2,:] @ C[:,5] = d23
acc0 -= (d23)
acc1 += (d23)
acc2 -= (d23)
acc4 += (d23)

# t3 = d[3,:] @ C[:,5] = d33
acc1 -= (d33)
acc2 -= (d33)
acc5 += (d33)

# t4 = d[4,:] @ C[:,5] = d43
acc2 += (d43)

# Agora acc0..acc5 = D[:,5]
# Calcula y = A.T @ (D[:,5] ⊙ G[:,5])
y0 = F(0)
y1 = F(0)
y2 = F(0)
y0 += acc0 * G05
y0 += acc3 * G35
y0 += acc4 * G45
y1 += acc1 * G15
y1 += acc3 * G35
y1 += acc5 * G55
y2 += acc2 * G25
y2 += acc4 * G45
y2 += acc5 * G55

# Acumula s += y @ A[5,:]
s01 += y0
s02 += y0
s11 += y1
s12 += y1
s21 += y2
s22 += y2

# ============================================================
# RESULTADO POR REGISTRADORES
# ============================================================
s_reg = np.array([
    [s00, s01, s02],
    [s10, s11, s12],
    [s20, s21, s22],
], dtype=object)

print("s_reg =")
print(s_reg)


## 4. Verificação

Compara a versão por registradores com a forma matricial direta.


In [ ]:
# Reconstrói as matrizes a partir dos registradores apenas para conferir.
d_reg = np.array([
    [d00, d01, d02, d03, d04],
    [d10, d11, d12, d13, d14],
    [d20, d21, d22, d23, d24],
    [d30, d31, d32, d33, d34],
    [d40, d41, d42, d43, d44],
], dtype=object)

G_reg = np.array([
    [G00, G01, G02, G03, G04, G05],
    [G10, G11, G12, G13, G14, G15],
    [G20, G21, G22, G23, G24, G25],
    [G30, G31, G32, G33, G34, G35],
    [G40, G41, G42, G43, G44, G45],
    [G50, G51, G52, G53, G54, G55],
], dtype=object)

D_direct = CT @ d_reg @ C
S_direct = D_direct * G_reg
s_direct = AT @ S_direct @ A

print("D_direct =")
print(D_direct)
print()
print("G_reg =")
print(G_reg)
print()
print("s_direct =")
print(s_direct)
print()
print("s_reg == s_direct?", np.array_equal(s_reg, s_direct))
print("s_reg == s_ref?", np.array_equal(s_reg, s_ref))


## 5. Versão funcional compacta por coluna

Esta célula mostra o mesmo algoritmo em forma compacta, útil para parametrizar e comparar ciclos.

Ela mantém a mesma ideia:

\[
D_{:,j}=C^T(dC_{:,j})
\]

\[
S_{:,j}=D_{:,j}\odot G_{:,j}
\]

\[
s \mathrel{+}= (A^T S_{:,j})A_{j,:}
\]


In [ ]:
def if2d_stream_by_column(d, g, AT, B, CT):
    A = AT.T
    C = CT.T

    G = B @ g @ B.T
    s = np.zeros((AT.shape[0], AT.shape[0]), dtype=object)

    for j in range(B.shape[0]):
        # coluna atual de D
        acc = CT @ (d @ C[:, j])

        # coluna atual de S
        S_col = acc * G[:, j]

        # y = A.T @ S[:,j], mas A.T aqui é AT
        y = AT @ S_col

        # s += y @ A[j,:]
        s += np.outer(y, A[j, :])

    return s

s_stream = if2d_stream_by_column(d_np, g_np, AT, B, CT)

print("s_stream =")
print(s_stream)
print()
print("s_stream == s_ref?", np.array_equal(s_stream, s_ref))


## 6. Contagem de registradores

### Datapath online com \(G\) pré-computado

\[
d[5\times5] + D_{\text{col}}[6] + y[3] + s[3\times3]
\]

\[
25 + 6 + 3 + 9 = 43
\]

### Se \(G\) também for registrador

\[
43 + 36 = 79
\]

### Se \(S\) fosse materializado

Adicionar \(36\) registradores para \(S\), o que geralmente não é necessário.

### Se \(D\) fosse materializado

Adicionar \(36\) registradores para \(D\), que é justamente o que esta versão evita.


In [ ]:
print("Contagem de registradores:")
print("d[5x5]      = 25")
print("D_col[6]    = 6")
print("y[3]        = 3")
print("s[3x3]      = 9")
print("Total sem G = 43")
print("G[6x6]      = 36")
print("Total com G = 79")
